# 34 — Línea base de OpenKOS 0.7.3: ¿el cambio de versión altera la extracción?

**Autor:** Jason Sepúlveda S.
**Proyecto:** Evaluación experimental de la extracción automática y la representación
persistente de decisiones organizacionales frente a Recuperación Aumentada (RAG)
**Depende de:** [ADR 0026](../docs/decisions/0026-adr-extension-extractor-con-modelos-hosted.md) (enmiendas primera y segunda) ·
[ADR 0011](../docs/decisions/0011-adr-alineacion-rol-e-intervalos-del-experimento-a.md) ·
[ADR 0012](../docs/decisions/0012-adr-umbral-tau-del-experimento-a.md) ·
[notebook 28](28-jss-experimento-a.ipynb) · [notebook 33](33-jss-extraccion-con-modelos-de-pago.ipynb) ·
[protocolo de la extensión](../docs/protocols/extraccion-hosted.md)

**Alcance.** Registro de la corrida `experimento-a-base-0-7-3` (issue #331): `qwen3:8b` con OpenKOS 0.7.3 sobre
las 11 series de evaluación, 3 repeticiones por serie, con la semilla y la temperatura de la corrida formal.
Se compara con la corrida formal `experimento-a-formal` (OpenKOS 0.7.0). El notebook lee las corridas guardadas en
`reports/experiment-a/`; **no llama a ningún modelo generativo**. Toda cifra sale de funciones de
`src/afg/experiments/` con test (`hosted_comparison.py` y `hosted_notebook.py`); el notebook solo las
invoca y las muestra.

**Estado.** Descriptivo (ADR 0026, decisión 9): las series de evaluación ya se usaron en la corrida formal, de
modo que ningún resultado de este notebook confirma una hipótesis de la tesis. **Requisito:** el cálculo de
similitudes necesita Ollama en marcha con el modelo de embeddings `bge-m3`.

> **Aviso de alcance (ADR 0005).** Las series son series de **evaluación**. El notebook muestra solo
> **identificadores de serie y de reunión, nombres de modelo y cifras agregadas**. No muestra ni cita frases de
> decisión ni contenido de transcripciones. No contiene ninguna cifra del Experimento B.

# 1. Para qué sirve esta corrida

OpenKOS 0.7.3 **no se hizo para mejorar la extracción**. La versión nueva agrega adaptadores para que OpenKOS
funcione con modelos de pago a través de APIs: un *backend* compatible con OpenAI, el registro de los tokens
de cada `ingest` y el campo `extraction_status`. La extensión del Experimento A (ADR 0026) corre esos modelos de
pago con la 0.7.3, mientras que la corrida formal usó la 0.7.0.

Para que la comparación entre un modelo de pago y la corrida formal tenga el mismo piso, hay que comprobar que
**el cambio de versión, por sí solo, no altera la extracción**. Por eso se corrió `qwen3:8b` (el modelo de la
corrida formal) sobre la 0.7.3, con la misma semilla y la misma temperatura. Si las cifras coinciden, la
diferencia que se observe con un modelo de pago se puede atribuir al modelo y no a la versión.

**Qué se hace.** (1) Se muestra la identidad de la corrida a partir de su manifiesto. (2) Se comprueba que está
completa. (3) Se calcula el *efecto de versión*: la diferencia pareada por reunión entre esta corrida y la
formal, con intervalos bootstrap. (4) Se compara el tiempo por reunión. (5) Se retoma el puente de la 0.7.2.

In [1]:
import warnings

import pandas as pd
from IPython.display import Markdown, display

from afg.cli import make_alignment_similarity
from afg.evaluation.reference import load_series_reference
from afg.experiments.hosted_comparison import build_comparison, load_run_data
from afg.experiments.hosted_extraction import load_hosted_config
from afg.experiments.hosted_notebook import (
    bridge_frame,
    bridge_state,
    format_es,
    format_interval,
    local_baseline_state,
)
from afg.shared.config import load_corpus_config
from afg.shared.paths import EXPERIMENT_A_DIR, OPENKOS_WORKSPACES_DIR

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

config = load_hosted_config()
spec = config.comparison
EVALUATION_SERIES = load_corpus_config()["oe1"]["evaluation_series"]
TAU_KEYS = [f"{t:.2f}" for t in (spec.tau, *spec.sensitivity_taus)]
local_name = config.local_baseline.run_name
local_info = local_baseline_state(
    config, runs_root=EXPERIMENT_A_DIR, evaluation_series=EVALUATION_SERIES
)
display(
    Markdown(
        f"Estado de la corrida `{local_name}`: **{local_info['state']}**. {local_info['detail']}"
    )
)

Estado de la corrida `experimento-a-base-0-7-3`: **lista**. La segunda línea base terminó.

# 2. Identidad de la corrida

**Pregunta.** ¿Con qué código, versiones y ajustes se hizo la corrida?

**Qué se hace.** Se lee el manifiesto que la propia corrida escribió al terminar.

In [2]:
local = load_run_data(EXPERIMENT_A_DIR / local_name, OPENKOS_WORKSPACES_DIR / local_name)
manifest = local.manifest
kos = manifest["openkos"]
created = pd.Timestamp(manifest["created_at"])
updated = pd.Timestamp(manifest["updated_at"])
minutes = int((updated - created).total_seconds() // 60)
identity = pd.DataFrame(
    [
        ("Corrida", manifest["run_name"]),
        ("Commit del código", manifest["git_commit"][:12]),
        ("Árbol de trabajo con cambios (git_dirty)", str(manifest["git_dirty"]).lower()),
        ("Versión de OpenKOS", ", ".join(kos["versions"])),
        ("Ollama", manifest["ollama_version"]),
        ("Modelo generativo", kos["generative_model"]),
        ("Modelo de embeddings", kos["embedding_model"]),
        ("Semilla base", kos["seed"]),
        ("Temperatura", format_es(kos["temperature"], 1)),
        ("Series / repeticiones", f"{len(manifest['series_ids'])} / {manifest['runs']}"),
        ("Partición", manifest["partition"]),
        ("Duración (inicio a fin del manifiesto)", f"{minutes // 60} h {minutes % 60} min"),
    ],
    columns=["Campo", "Valor"],
)
display(identity)

,Campo,Valor
0,Corrida,experimento-a-base-0-7-3
1,Commit del código,735c3ca61aae
2,Árbol de trabajo con cambios (git_dirty),false
3,Versión de OpenKOS,0.7.3
4,Ollama,0.40.16
5,Modelo generativo,qwen3:8b
6,Modelo de embeddings,bge-m3
7,Semilla base,42
8,Temperatura,"0,1"
9,Series / repeticiones,11 / 3


In [3]:
formal_name = config.baseline_run_name
formal = load_run_data(EXPERIMENT_A_DIR / formal_name, OPENKOS_WORKSPACES_DIR / formal_name)
fk = formal.manifest["openkos"]
same = {
    "modelo": fk["generative_model"] == kos["generative_model"],
    "semilla": fk["seed"] == kos["seed"],
    "temperatura": fk["temperature"] == kos["temperature"],
}
display(
    Markdown(
        f"**Lectura.** La corrida formal usó OpenKOS {', '.join(fk['versions'])} y esta usa "
        f"{', '.join(kos['versions'])}. Coinciden el modelo ({same['modelo']}), la semilla "
        f"({same['semilla']}) y la temperatura ({same['temperatura']}). Esta corrida se hizo "
        f"con Ollama "
        f"{manifest['ollama_version']}; la formal, con otra versión del servidor local, de modo "
        f"que el "
        f"efecto de versión mide **también** ese cambio de infraestructura (cuarta enmienda del "
        f"ADR 0026), "
        f"y no solo el de OpenKOS."
    )
)

**Lectura.** La corrida formal usó OpenKOS 0.7.0 y esta usa 0.7.3. Coinciden el modelo (True), la semilla (True) y la temperatura (True). Esta corrida se hizo con Ollama 0.40.16; la formal, con otra versión del servidor local, de modo que el efecto de versión mide **también** ese cambio de infraestructura (cuarta enmienda del ADR 0026), y no solo el de OpenKOS.

# 3. Completitud

**Pregunta.** ¿Terminaron todas las extracciones, sin errores y con las citas de las decisiones en su lugar?

**Qué se hace.** Se cuentan los pares serie × repetición, y se suman los contadores de citas y de decisiones sin
mapear que OpenKOS guardó en cada registro de `extracciones.jsonl`.

In [4]:
payload = build_comparison(
    config=config,
    labels=[],
    include_bridge=bridge_state(config, runs_root=EXPERIMENT_A_DIR)["state"] == "lista",
    pilot=False,
    runs_root=EXPERIMENT_A_DIR,
    workspaces_root=OPENKOS_WORKSPACES_DIR,
    reference_of=lambda series_id: load_series_reference(series_id, language="en").decisions,
    similarity=make_alignment_similarity(),
    baseline="local",
)
effect = payload["version_effect"]
operation = effect["operation"]
quotes = {"total": 0, "found": 0, "not_found": 0}
unmapped = 0
for extraction in local.runs:
    for key in quotes:
        quotes[key] += extraction.record["quotes"][key]
    unmapped += len(extraction.record["unmapped"])
completeness = operation["completeness"]["candidate"]
status_counts = operation["extraction_status"]["candidate"]
display(
    pd.DataFrame(
        [
            ("Pares serie por repetición esperados", effect["runs_expected"]),
            ("Pares terminados (ok)", effect["runs_ok"]),
            ("Pares fallidos", effect["runs_failed"]),
            ("Series cubiertas", effect["n_series"]),
            ("Reuniones", effect["n_meetings"]),
            (
                "Extracciones reunión por repetición completas",
                f"{completeness['complete']} de {completeness['expected']}",
            ),
            ("Extracciones con estado distinto de completo", sum(status_counts.values())),
            ("Decisiones sin mapear", unmapped),
            ("Citas verificadas / citas totales", f"{quotes['found']} / {quotes['total']}"),
            ("Citas no encontradas", quotes["not_found"]),
        ],
        columns=["Verificación", "Valor"],
    )
)
display(
    Markdown(
        f"**Lectura.** Terminaron {effect['runs_ok']} de {effect['runs_expected']} pares, con "
        f"{effect['runs_failed']} fallidos; en {effect['n_meetings']} reuniones por repetición "
        f"hubo "
        f"{unmapped} decisiones sin mapear y {quotes['not_found']} citas no encontradas."
    )
)

,Verificación,Valor
0,Pares serie por repetición esperados,33
1,Pares terminados (ok),33
2,Pares fallidos,0
3,Series cubiertas,11
4,Reuniones,44
5,Extracciones reunión por repetición completas,132 de 132
6,Extracciones con estado distinto de completo,0
7,Decisiones sin mapear,0
8,Citas verificadas / citas totales,1738 / 1738
9,Citas no encontradas,0


**Lectura.** Terminaron 33 de 33 pares, con 0 fallidos; en 44 reuniones por repetición hubo 0 decisiones sin mapear y 0 citas no encontradas.

# 4. Efecto de versión: calidad de la extracción

**Pregunta.** Con τ = 0,70, ¿cambian la precision, el recall y el F1 al pasar de OpenKOS 0.7.0 a 0.7.3?

**Qué se hace.** Se calcula la diferencia (0.7.3 menos 0.7.0) con un bootstrap pareado por reunión, con los
mismos remuestreos, la misma semilla y el mismo nivel Bonferroni de la extensión (por los seis modelos de pago,
aunque esta fila no entra en esa familia). Una diferencia cuyo intervalo cruza el 0 no se puede distinguir de
cero. **Qué no se afirma:** que no haya diferencia, sino que con 44 reuniones no se demuestra una.

In [5]:
resamples = format(spec.bootstrap_resamples, ",").replace(",", ".")
level = format_es(100 * effect["interval_level"], 2)
display(
    Markdown(
        f"Bootstrap pareado por reunión, {resamples} remuestreos, semilla {spec.bootstrap_seed}, "
        f"intervalos de {level} % (Bonferroni por {effect['n_models']} modelos)."
    )
)
table_rows = []
for key in TAU_KEYS[:1]:
    item = effect["taus"][key]
    row = {"Medida": "Valor 0.7.3", "τ": format_es(item["tau"])}
    for name, title in (("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")):
        row[title] = format_es(item["candidate"][name], 3)
    table_rows.append(row)
    row = {"Medida": "Valor formal (0.7.0)", "τ": format_es(item["tau"])}
    for name, title in (("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")):
        row[title] = format_es(item["baseline"][name], 3)
    table_rows.append(row)
    row = {"Medida": "Diferencia (0.7.3 menos 0.7.0)", "τ": format_es(item["tau"])}
    for name, title in (("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")):
        diff = item["diff"][name]
        row[title] = format_interval(diff["estimate"], diff["interval"], 3, signed=True)
    table_rows.append(row)
display(pd.DataFrame(table_rows))

Bootstrap pareado por reunión, 10.000 remuestreos, semilla 42, intervalos de 99,17 % (Bonferroni por 6 modelos).

,Medida,τ,Precision,Recall,F1
0,Valor 0.7.3,"0,70","0,147","0,306","0,198"
1,Valor formal (0.7.0),"0,70","0,149","0,308","0,201"
2,Diferencia (0.7.3 menos 0.7.0),"0,70","-0,002 [-0,015; +0,010]","-0,002 [-0,031; +0,022]","-0,002 [-0,020; +0,013]"


In [6]:
entry = effect["taus"][TAU_KEYS[0]]
shown = []
for name, title in (("precision", "precision"), ("recall", "recall"), ("f1", "F1")):
    low, high = entry["diff"][name]["interval"]
    where = "cruza el 0" if low <= 0 <= high else ("sobre 0" if low > 0 else "bajo 0")
    figure = format_interval(
        entry["diff"][name]["estimate"], entry["diff"][name]["interval"], digits=3, signed=True
    )
    shown.append(f"{title} {figure} ({where})")
base, cand = entry["baseline"], entry["candidate"]
display(
    Markdown(
        f"**Lectura.** Con τ = {format_es(spec.tau)}: formal (0.7.0) precision "
        f"{format_es(base['precision'], 3)}, "
        f"recall {format_es(base['recall'], 3)}, F1 {format_es(base['f1'], 3)}; esta corrida "
        f"(0.7.3) "
        f"precision {format_es(cand['precision'], 3)}, recall {format_es(cand['recall'], 3)}, "
        f"F1 {format_es(cand['f1'], 3)}. Diferencias: " + "; ".join(shown) + f". Veredicto: "
        f"**{entry['verdict']}**."
    )
)

**Lectura.** Con τ = 0,70: formal (0.7.0) precision 0,149, recall 0,308, F1 0,201; esta corrida (0.7.3) precision 0,147, recall 0,306, F1 0,198. Diferencias: precision -0,002 [-0,015; +0,010] (cruza el 0); recall -0,002 [-0,031; +0,022] (cruza el 0); F1 -0,002 [-0,020; +0,013] (cruza el 0). Veredicto: **sin diferencia demostrada**.

## Sensibilidad al umbral τ

La regla del ADR 0012 fija τ = 0,70 como criterio principal; 0,75 y 0,80 son sensibilidades. Se muestran los tres
para no ocultar que, con un umbral más exigente, el resultado puede cambiar.

In [7]:
rows = []
for key in TAU_KEYS:
    item = effect["taus"][key]
    row = {"τ": format_es(item["tau"])}
    for name, title in (("precision", "Δ precision"), ("recall", "Δ recall"), ("f1", "Δ F1")):
        diff = item["diff"][name]
        row[title] = format_interval(diff["estimate"], diff["interval"], 3, signed=True)
    row["Veredicto"] = item["verdict"]
    rows.append(row)
display(pd.DataFrame(rows))
strict = effect["taus"][TAU_KEYS[-1]]
display(
    Markdown(
        f"**Lectura.** Con el criterio principal (τ = {format_es(spec.tau)}) el veredicto es "
        f"«{effect['taus'][TAU_KEYS[0]]['verdict']}». Con τ = {format_es(strict['tau'])} el "
        f"veredicto es "
        f"«{strict['verdict']}»: las diferencias son pequeñas (del orden de un punto "
        f"porcentual) pero sus "
        f"intervalos quedan bajo 0. Se informa tal cual: es una sensibilidad y no cambia la "
        f"lectura "
        f"principal, pero muestra que un cambio fino de comportamiento entre versiones no se "
        f"descarta."
    )
)

,τ,Δ precision,Δ recall,Δ F1,Veredicto
0,"0,70","-0,002 [-0,015; +0,010]","-0,002 [-0,031; +0,022]","-0,002 [-0,020; +0,013]",sin diferencia demostrada
1,"0,75","-0,008 [-0,018; +0,000]","-0,016 [-0,035; +0,001]","-0,011 [-0,023; +0,000]",sin diferencia demostrada
2,"0,80","-0,006 [-0,013; -0,000]","-0,012 [-0,025; -0,001]","-0,008 [-0,017; -0,001]",empeora


**Lectura.** Con el criterio principal (τ = 0,70) el veredicto es «sin diferencia demostrada». Con τ = 0,80 el veredicto es «empeora»: las diferencias son pequeñas (del orden de un punto porcentual) pero sus intervalos quedan bajo 0. Se informa tal cual: es una sensibilidad y no cambia la lectura principal, pero muestra que un cambio fino de comportamiento entre versiones no se descarta.

# 5. Tiempo por reunión

**Pregunta.** ¿Tarda distinto el `ingest` de una reunión con la 0.7.3?

**Qué se hace.** Se compara la mediana del tiempo de `ingest` por reunión (de los manifiestos de los espacios de
trabajo) y se calcula la razón 0.7.3 / 0.7.0 con su intervalo bootstrap. El tiempo depende también de la
carga del equipo y de la versión de Ollama, por lo que es una referencia, no una medida limpia.

In [8]:
time = operation["time"]
ratio = time["ratio"]
display(
    pd.DataFrame(
        [
            ("Mediana por reunión, formal (0.7.0)", f"{format_es(time['baseline_median_s'], 1)} s"),
            (
                "Mediana por reunión, esta corrida (0.7.3)",
                f"{format_es(time['candidate_median_s'], 1)} s",
            ),
            (
                "Razón 0.7.3 / 0.7.0",
                format_interval(ratio["estimate"], ratio["interval"], signed=False),
            ),
            ("Reuniones comparadas", time["n_meetings"]),
            ("Veredicto", time["verdict"]),
        ],
        columns=["Medida", "Valor"],
    )
)
dpm = operation["decisions_per_meeting"]
display(
    Markdown(
        f"**Lectura.** La mediana pasa de {format_es(time['baseline_median_s'], 1)} s a "
        f"{format_es(time['candidate_median_s'], 1)} s por reunión; el veredicto de tiempo es "
        f"«{time['verdict']}». Las decisiones extraídas por reunión (media) son "
        f"{format_es(dpm['baseline']['mean'], 1)} con la 0.7.0 y "
        f"{format_es(dpm['candidate']['mean'], 1)} "
        f"con la 0.7.3."
    )
)

,Medida,Valor
0,"Mediana por reunión, formal (0.7.0)","307,1 s"
1,"Mediana por reunión, esta corrida (0.7.3)","327,5 s"
2,Razón 0.7.3 / 0.7.0,"1,09 [1,00; 1,19]"
3,Reuniones comparadas,44
4,Veredicto,sin diferencia demostrada


**Lectura.** La mediana pasa de 307,1 s a 327,5 s por reunión; el veredicto de tiempo es «sin diferencia demostrada». Las decisiones extraídas por reunión (media) son 13,1 con la 0.7.0 y 13,2 con la 0.7.3.

# 6. El puente de la 0.7.2

**Pregunta.** ¿Qué pasa con el puente (`experimento-a-puente-0-7-2`, ES2002 × 1) que salió fuera del rango
formal?

**Qué se hace.** Se repite la tabla de la sección 3 del notebook 33. El puente es una sola corrida de una sola
serie, con OpenKOS 0.7.2; la primera enmienda del ADR 0026 ya anticipó que esa evidencia es gruesa y que decide la
línea base de la misma versión, que es esta corrida.

In [9]:
bridge = payload["bridge"]
if bridge is not None and bridge["status"] == "ok":
    display(bridge_frame(bridge))
    outside = {
        series: item["outside_metrics"]
        for series, item in bridge["series"].items()
        if item.get("inside") is False
    }
    display(
        Markdown(
            f"**Lectura.** El puente queda fuera del rango formal en: {outside}. Es un hecho ya "
            f"conocido y "
            f"no se reinterpreta aquí: una sola corrida no separa el efecto de la versión del "
            f"azar, y por eso "
            f"se corrió esta línea base sobre las once series y tres repeticiones. Con ella, el "
            f"efecto de "
            f"versión de la sección 4 es el que decide."
        )
    )
else:
    display(Markdown("**Pendiente.** La corrida del puente no está disponible."))

,Serie,Métrica,Puente,Rango formal,Dentro
0,ES2002,precision,"0,21","0,19 a 0,20",no
1,ES2002,recall,"0,35","0,32 a 0,35",sí
2,ES2002,f1,"0,27","0,24 a 0,26",no


**Lectura.** El puente queda fuera del rango formal en: {'ES2002': ['precision', 'f1']}. Es un hecho ya conocido y no se reinterpreta aquí: una sola corrida no separa el efecto de la versión del azar, y por eso se corrió esta línea base sobre las once series y tres repeticiones. Con ella, el efecto de versión de la sección 4 es el que decide.

# 7. Conclusión

In [10]:
main, last = effect["taus"][TAU_KEYS[0]], effect["taus"][TAU_KEYS[-1]]
diffs = [main["diff"][m]["estimate"] for m in ("precision", "recall", "f1")]
crossing = all(
    main["diff"][m]["interval"][0] <= 0 <= main["diff"][m]["interval"][1]
    for m in ("precision", "recall", "f1")
)
display(
    Markdown(
        f"- **Corrida completa.** Terminaron {effect['runs_ok']} de {effect['runs_expected']} "
        f"pares, "
        f"con {effect['runs_failed']} fallos, sobre {effect['n_meetings']} reuniones de "
        f"{effect['n_series']} series; hubo {unmapped} decisiones sin mapear y "
        f"{quotes['not_found']} "
        f"citas no encontradas.\n"
        f"- **El cambio de versión no muestra efecto en la extracción.** Con τ = "
        f"{format_es(spec.tau)}, "
        f"la precision, el recall y el F1 de `qwen3:8b` con OpenKOS 0.7.3 difieren de la "
        f"corrida formal "
        f"en {', '.join(format_es(d, 3, signed=True) for d in diffs)} (en ese orden); "
        f"{'los tres intervalos cruzan el 0' if crossing else 'algún intervalo no cruza el 0'}. "
        f"Veredicto: **{effect['verdict']}**. El tiempo por reunión: "
        f"**{time['verdict']}**.\n"
        f"- **Matiz.** Con τ = {format_es(last['tau'])} el veredicto es «{last['verdict']}», con "
        f"diferencias muy pequeñas (sensibilidad, no criterio). El puente de la 0.7.2 quedó "
        f"fuera del "
        f"rango formal en {', '.join(outside['ES2002'])}, pero es una sola corrida: decide el "
        f"efecto de "
        f"versión sobre las once series (ADR 0026, primera enmienda).\n"
        f"- **Qué significa para la extensión.** Los modelos de pago se comparan con la corrida "
        f"formal y "
        f"con esta línea base con el mismo piso: las diferencias que aparezcan se leen como "
        f"efecto del "
        f"modelo. Esta línea base mide, además del cambio de OpenKOS, el de Ollama.\n"
        f"- **Próximo paso.** Pilotos y corridas completas de los seis modelos de pago (issues "
        f"#332 y "
        f"#333), comparados con esta línea base y con la corrida formal en el notebook 33."
    )
)

- **Corrida completa.** Terminaron 33 de 33 pares, con 0 fallos, sobre 44 reuniones de 11 series; hubo 0 decisiones sin mapear y 0 citas no encontradas.
- **El cambio de versión no muestra efecto en la extracción.** Con τ = 0,70, la precision, el recall y el F1 de `qwen3:8b` con OpenKOS 0.7.3 difieren de la corrida formal en -0,002, -0,002, -0,002 (en ese orden); los tres intervalos cruzan el 0. Veredicto: **sin diferencia demostrada**. El tiempo por reunión: **sin diferencia demostrada**.
- **Matiz.** Con τ = 0,80 el veredicto es «empeora», con diferencias muy pequeñas (sensibilidad, no criterio). El puente de la 0.7.2 quedó fuera del rango formal en precision, f1, pero es una sola corrida: decide el efecto de versión sobre las once series (ADR 0026, primera enmienda).
- **Qué significa para la extensión.** Los modelos de pago se comparan con la corrida formal y con esta línea base con el mismo piso: las diferencias que aparezcan se leen como efecto del modelo. Esta línea base mide, además del cambio de OpenKOS, el de Ollama.
- **Próximo paso.** Pilotos y corridas completas de los seis modelos de pago (issues #332 y #333), comparados con esta línea base y con la corrida formal en el notebook 33.